In [1]:
from sklearn.svm import SVC
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler

# Memuat data untuk contoh dasar
cancer = load_breast_cancer()
X_train, X_test, y_train, y_test = train_test_split(
    cancer.data, cancer.target, random_state=0)

** Prapemrosesan Manual (Langkah Dasar)**

In [2]:
# Menghitung scaling pada data training
scaler = MinMaxScaler().fit(X_train)
# Mentransformasi data training
X_train_scaled = scaler.transform(X_train)

# Melatih model SVM
svm = SVC()
svm.fit(X_train_scaled, y_train)

# Mengevaluasi model pada data test yang sudah di-scale
X_test_scaled = scaler.transform(X_test)
print(f"Test score: {svm.score(X_test_scaled, y_test):.2f}")

Test score: 0.97


**Bahaya Grid Search di Luar Pipeline (Salah)**

In [3]:
from sklearn.model_selection import GridSearchCV

# Prapemrosesan dilakukan sebelum CV (Data Leakage!)
scaler = MinMaxScaler().fit(X_train)
X_train_scaled = scaler.transform(X_train)

param_grid = {'C': [0.001, 0.01, 0.1, 1, 10, 100],
              'gamma': [0.001, 0.01, 0.1, 1, 10, 100]}
grid = GridSearchCV(SVC(), param_grid=param_grid, cv=5)
grid.fit(X_train_scaled, y_train)

print(f"Best cross-validation accuracy: {grid.best_score_:.2f}")

Best cross-validation accuracy: 0.98


Melakukan scaling pada seluruh data latihan sebelum cross-validation menyebabkan informasi bocor (data leakage) dari lipatan validasi ke lipatan latihan, sehingga hasil evaluasi menjadi terlalu optimis

**Membangun Pipeline Pertama**

In [4]:
from sklearn.pipeline import Pipeline

# Daftar langkah: (nama, instansi_estimator)
pipe = Pipeline([("scaler", MinMaxScaler()), ("svm", SVC())])

Langkah pertama adalah scaler dan langkah terakhir adalah model klasifikasi.

**Melatih Pipeline**

In [5]:
pipe.fit(X_train, y_train)

Pipeline(steps=[('scaler', MinMaxScaler()), ('svm', SVC())])

Saat memanggil fit, pipeline secara otomatis melakukan fit dan transform pada langkah prapemrosesan, lalu melatih model terakhir menggunakan data yang sudah ditransformasi.

**Mengevaluasi Pipeline**

In [6]:
print(f"Test score: {pipe.score(X_test, y_test):.2f}")

Test score: 0.97


Memanggil score pada pipeline akan mentransformasi data uji menggunakan scaler yang sudah dilatih pada data training, kemudian menghitung akurasi model


**Menentukan Grid Parameter untuk Pipeline**

In [7]:
# Nama parameter harus diawali dengan 'nama_step__' (double underscore)
param_grid = {'svm__C': [0.001, 0.01, 0.1, 1, 10, 100],
              'svm__gamma': [0.001, 0.01, 0.1, 1, 10, 100]}

Untuk menyetel parameter model di dalam pipeline menggunakan GridSearchCV, kita harus menggunakan sintaks khusus berupa nama langkah diikuti garis bawah ganda (__) dan nama parameternya


**GridSearchCV dengan Pipeline (Cara yang Benar)**

In [8]:
grid = GridSearchCV(pipe, param_grid=param_grid, cv=5)
grid.fit(X_train, y_train)

print(f"Best cross-validation accuracy: {grid.best_score_:.2f}")
print(f"Test set score: {grid.score(X_test, y_test):.2f}")
print(f"Best parameters: {grid.best_params_}")

Best cross-validation accuracy: 0.98
Test set score: 0.97
Best parameters: {'svm__C': 1, 'svm__gamma': 1}


 prapemrosesan dilakukan di dalam setiap iterasi cross-validation hanya pada lipatan data latihan, sehingga tidak ada kebocoran data dan hasil evaluasi menjadi jujur

**Menunjukkan Efek Kebocoran Data (Informasi Leakage)**

In [9]:
import numpy as np
rnd = np.random.RandomState(seed=0)
X = rnd.normal(size=(100, 10000))
y = rnd.normal(size=(100,))

Secara teoritis, model tidak boleh bisa mempelajari apa pun dari data ini.

**Seleksi Fitur Tanpa Pipeline (Hasil yang Menyesatkan)**

In [10]:
from sklearn.feature_selection import SelectPercentile, f_regression
from sklearn.linear_model import Ridge
from sklearn.model_selection import cross_val_score

# Seleksi fitur dilakukan pada seluruh dataset SEBELUM CV
select = SelectPercentile(score_func=f_regression, percentile=5).fit(X, y)
X_selected = select.transform(X)

print(f"Cross-validation score (outside pipeline): {np.mean(cross_val_score(Ridge(), X_selected, y, cv=5)):.2f}")

Cross-validation score (outside pipeline): 0.91


Karena seleksi fitur melihat hubungan dengan target pada seluruh data, ia menemukan "pola" kebetulan. Hasil CV menunjukkan akurasi tinggi (sekitar 0.91), padahal datanya acak. Ini adalah bukti nyata kebocoran data

**Seleksi Fitur dengan Pipeline (Hasil yang Akurat)**

In [11]:
pipe = Pipeline([("select", SelectPercentile(score_func=f_regression, percentile=5)),
                 ("ridge", Ridge())])
print(f"Cross-validation score (inside pipeline): {np.mean(cross_val_score(pipe, X, y, cv=5)):.2f}")

Cross-validation score (inside pipeline): -0.25


 Saat seleksi fitur dimasukkan ke dalam pipeline, ia hanya bisa "melihat" data latihan di setiap lipatan CV. Hasilnya menunjukkan skor negatif (buruk), yang mencerminkan realitas bahwa data tersebut acak

**Membuat Pipeline dengan make_pipeline**

In [12]:
from sklearn.pipeline import make_pipeline

# Nama langkah akan dibuat otomatis menjadi huruf kecil dari nama kelas
pipe_short = make_pipeline(MinMaxScaler(), SVC(C=100))
print(f"Pipeline steps: {pipe_short.steps}")

Pipeline steps: [('minmaxscaler', MinMaxScaler()), ('svc', SVC(C=100))]


make_pipeline adalah fungsi praktis yang tidak memerlukan penamaan langkah secara manua

**Penamaan Otomatis pada make_pipeline**

In [13]:
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

pipe_long = make_pipeline(StandardScaler(), PCA(n_components=2), StandardScaler())
print(f"Steps: {[step for step in pipe_long.steps]}")

Steps: [('standardscaler-1', StandardScaler()), ('pca', PCA(n_components=2)), ('standardscaler-2', StandardScaler())]


Jika ada kelas yang sama digunakan dua kali, scikit-learn akan menambahkan angka di belakangnya (misalnya standardscaler-1 dan standardscaler-2)

**Mengakses Atribut Langkah dengan named_steps**

In [14]:
pipe_long.fit(cancer.data)
# Mengakses komponen PCA
components = pipe_long.named_steps["pca"].components_
print(f"PCA components shape: {components.shape}")

PCA components shape: (2, 30)


Kita bisa menggunakan atribut named_steps (yang berupa dictionary) untuk mengambil informasi dari langkah tertentu di dalam pipeline setelah dilatih

**Pipeline dalam GridSearch dengan StandardScaler**

In [15]:
from sklearn.linear_model import LogisticRegression

pipe = make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000))
param_grid = {'logisticregression__C': [0.01, 0.1, 1, 10, 100]}

grid = GridSearchCV(pipe, param_grid, cv=5)
grid.fit(X_train, y_train)

GridSearchCV(cv=5,
             estimator=Pipeline(steps=[('standardscaler', StandardScaler()),
                                       ('logisticregression',
                                        LogisticRegression(max_iter=1000))]),
             param_grid={'logisticregression__C': [0.01, 0.1, 1, 10, 100]})

Contoh penggunaan make_pipeline dalam GridSearchCV. Perhatikan nama parameter logisticregression__C yang diambil dari nama kelas model

** Mengakses Model Terbaik dalam Grid Search**

In [16]:
# Mengambil model regresi logistik terbaik dari grid search
best_lr = grid.best_estimator_.named_steps["logisticregression"]
print(f"Logistic regression coefficients: {best_lr.coef_}")

Logistic regression coefficients: [[-0.31167303 -0.58082201 -0.32131835 -0.38161278 -0.11923966  0.43130513
  -0.70867977 -0.85378868 -0.46682033  0.11842553 -1.384584    0.08915178
  -0.95504656 -0.93809826  0.18173417  0.99841869  0.1098606  -0.34148205
   0.20112256  0.80467192 -0.91482867 -0.91731629 -0.81023153 -0.85401188
  -0.45736929  0.11351219 -0.8359122  -0.98702282 -0.59104801 -0.62212143]]


**Grid-Searching Prapemrosesan (Polynomial Features)**

In [17]:
from sklearn.preprocessing import PolynomialFeatures

pipe = Pipeline([("scaler", StandardScaler()),
                 ("poly", PolynomialFeatures()),
                 ("ridge", Ridge())])

param_grid = {'poly__degree': [20-22],
              'ridge__alpha': [0.001, 0.01, 0.1, 1, 10, 100]}

Salah satu kekuatan utama pipeline adalah kemampuan untuk mencari parameter prapemrosesan (seperti derajat polinomial) bersamaan dengan parameter model

** Grid-Searching Memilih Model yang Digunakan**

In [24]:
from sklearn.ensemble import RandomForestClassifier

# Pipeline awal sebagai template
pipe = Pipeline([('preprocessing', StandardScaler()), ('classifier', SVC())])

# Daftar dictionary untuk membandingkan model berbeda
param_grid = [
    {'classifier': [SVC()], 'preprocessing': [StandardScaler(), None],
     'classifier__gamma': [0.001, 0.01, 0.1, 1, 10, 100],
     'classifier__C': [0.001, 0.01, 0.1, 1, 10, 100]},
    {'classifier': [RandomForestClassifier(n_estimators=100)],
     'preprocessing': [None], 'classifier__max_features': [20-22]}
]

**Menjalankan Pemilihan Model Otomatis**

In [25]:
X_train, X_test, y_train, y_test = train_test_split(
    cancer.data, cancer.target, random_state=0)

grid = GridSearchCV(pipe, param_grid, cv=5)
grid.fit(X_train, y_train)

print(f"Best params:\n{grid.best_params_}")
print(f"Best cross-validation score: {grid.best_score_:.2f}")
print(f"Test-set score: {grid.score(X_test, y_test):.2f}")

Best params:
{'classifier': SVC(), 'classifier__C': 10, 'classifier__gamma': 0.01, 'preprocessing': StandardScaler()}
Best cross-validation score: 0.99
Test-set score: 0.98


/usr/local/lib/python3.12/dist-packages/sklearn/model_selection/_validation.py:528: FitFailedWarning: 
5 fits failed out of a total of 365.
The score on these train-test partitions for these parameters will be set to nan.
If these failures are not expected, you can try to debug them by setting error_score='raise'.

Below are more details about the failures:
--------------------------------------------------------------------------------
5 fits failed with the following error:
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/sklearn/model_selection/_validation.py", line 866, in _fit_and_score
    estimator.fit(X_train, y_train, **fit_params)
  File "/usr/local/lib/python3.12/dist-packages/sklearn/base.py", line 1389, in wrapper
    return fit_method(estimator, *args, **kwargs)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/sklearn/pipeline.py", line 662, in fit
    self._final_estimator.fit(Xt, y, **las

Hasil akhir menunjukkan model mana yang menang beserta konfigurasi prapemrosesannya, memberikan alur kerja yang sangat kuat dan otomatis.